# 09 Wetterverteilung analysieren

Dieses Notebook implementiert BD-19. Es prueft auf Match-Ebene, ob Temperatur, gefuehlte Temperatur und Regen genug Streuung fuer die folgenden Analysefragen haben.

Input:
- `data/gold/team_match_analysis_dataset.csv`

Outputs:
- `outputs/figures/weather_temperature_histogram.png`
- `outputs/figures/weather_feels_like_histogram.png`
- `outputs/figures/weather_temperature_by_tournament.png`
- `outputs/figures/rain_share.png`
- `outputs/tables/bd19_weather_distribution_summary.csv`

## Methodischer Ansatz

Das Gold-Dataset enthaelt pro Spiel zwei Team-Perspektiven. Fuer Wetterverteilungen wird deshalb zuerst auf eindeutige `match_id` reduziert, damit jedes Wetterereignis nur einmal zaehlt.

Die EDA ist DataFrame-orientiert aufgebaut: numerische Variablen werden mit `describe`, Histogrammen und Boxplots betrachtet; kategoriale Anteile entstehen ueber `groupby`. So bleiben die Verteilungen reproduzierbar und direkt an die tabellarischen Pipeline-Artefakte anschliessbar.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from project_paths import project_root

ROOT = project_root()
DATA_PATH = ROOT / 'data' / 'gold' / 'team_match_analysis_dataset.csv'
FIGURE_DIR = ROOT / 'outputs' / 'figures'
TABLE_DIR = ROOT / 'outputs' / 'tables'

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TEMPERATURE_HIST_PATH = FIGURE_DIR / 'weather_temperature_histogram.png'
FEELS_LIKE_HIST_PATH = FIGURE_DIR / 'weather_feels_like_histogram.png'
TEMPERATURE_BY_TOURNAMENT_PATH = FIGURE_DIR / 'weather_temperature_by_tournament.png'
RAIN_SHARE_PATH = FIGURE_DIR / 'rain_share.png'
SUMMARY_PATH = TABLE_DIR / 'bd19_weather_distribution_summary.csv'

required_columns = [
    'match_id',
    'short_name',
    'temperature_c',
    'feels_like_c',
    'rain_mm',
    'rain_flag',
    'weather_missing_flag',
]

## Daten laden und Match-Ebene bilden

Die Wetterdaten werden aus dem finalen Analyse-Dataset gelesen. Durch `drop_duplicates('match_id')` bleibt pro Spiel genau eine Wetterbeobachtung erhalten.

In [ ]:
assert DATA_PATH.exists(), f'Missing required input file: {DATA_PATH}'

team_match_df = pd.read_csv(DATA_PATH)
missing_columns = [column for column in required_columns if column not in team_match_df.columns]
assert not missing_columns, f'Missing required columns: {missing_columns}'

weather_df = (
    team_match_df[required_columns]
    .drop_duplicates(subset=['match_id'])
    .sort_values(['short_name', 'match_id'])
    .reset_index(drop=True)
)

assert len(weather_df) > 0, 'Weather dataset is empty after reducing to match level.'
assert weather_df['match_id'].is_unique, 'Expected one row per match after deduplication.'
assert not weather_df['weather_missing_flag'].any(), 'Weather data still contains missing weather rows.'

weather_df.head()

## Kennzahlen zur Wetterstreuung

Die Tabelle fasst zentrale Streuungsmasse zusammen. Sie hilft bei der Entscheidung, ob Wetter als Kontextfaktor sinnvoll weiter analysiert werden kann.

In [ ]:
summary_rows = []
for column in ['temperature_c', 'feels_like_c', 'rain_mm']:
    series = weather_df[column].dropna()
    summary_rows.append({
        'metric': column,
        'n_matches': int(series.count()),
        'min': round(float(series.min()), 2),
        'q25': round(float(series.quantile(0.25)), 2),
        'median': round(float(series.median()), 2),
        'mean': round(float(series.mean()), 2),
        'q75': round(float(series.quantile(0.75)), 2),
        'max': round(float(series.max()), 2),
        'std': round(float(series.std()), 2),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(SUMMARY_PATH, index=False)
summary

## Histogramm Temperatur

Das Histogramm zeigt, ob die beobachteten Spiele nur in einem engen Temperaturbereich liegen oder ob genug Streuung vorhanden ist.

In [ ]:
values = weather_df['temperature_c'].dropna()

fig, ax = plt.subplots(figsize=(10, 6), dpi=160)
ax.hist(values, bins=14, color='#3569a8', edgecolor='white')
ax.set_title('Verteilung der Temperatur bei Anpfiff', loc='left', fontweight='bold', pad=30)
ax.set_xlabel('Temperatur in Grad C')
ax.set_ylabel('Anzahl Spiele')
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
ax.text(
    0,
    1.01,
    f'n={len(values)} | min={values.min():.1f} | median={values.median():.1f} | max={values.max():.1f}',
    transform=ax.transAxes,
    color='#5f6368',
    fontsize=10,
)
fig.tight_layout()
fig.savefig(TEMPERATURE_HIST_PATH, bbox_inches='tight')
plt.show()

## Histogramm gefuehlte Temperatur

Die gefuehlte Temperatur beruecksichtigt Wettereffekte wie Wind und Luftfeuchtigkeit und kann daher von der gemessenen Temperatur abweichen.

In [ ]:
values = weather_df['feels_like_c'].dropna()

fig, ax = plt.subplots(figsize=(10, 6), dpi=160)
ax.hist(values, bins=14, color='#3569a8', edgecolor='white')
ax.set_title('Verteilung der gefuehlten Temperatur bei Anpfiff', loc='left', fontweight='bold', pad=30)
ax.set_xlabel('Gefuehlte Temperatur in Grad C')
ax.set_ylabel('Anzahl Spiele')
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
ax.text(
    0,
    1.01,
    f'n={len(values)} | min={values.min():.1f} | median={values.median():.1f} | max={values.max():.1f}',
    transform=ax.transAxes,
    color='#5f6368',
    fontsize=10,
)
fig.tight_layout()
fig.savefig(FEELS_LIKE_HIST_PATH, bbox_inches='tight')
plt.show()

## Boxplot Temperatur nach Turnier

Der Boxplot zeigt, ob einzelne Turniere systematisch waermer oder kuehler waren. Das ist wichtig, weil spaetere Modelle Turnier- und Wettereffekte nicht verwechseln sollten.

In [ ]:
tournament_order = list(weather_df['short_name'].drop_duplicates())
boxplot_values = [
    weather_df.loc[weather_df['short_name'] == tournament, 'temperature_c'].dropna()
    for tournament in tournament_order
]

fig, ax = plt.subplots(figsize=(11, 6), dpi=160)
box = ax.boxplot(boxplot_values, labels=tournament_order, patch_artist=True, showfliers=True)
for patch in box['boxes']:
    patch.set_facecolor('#9fbce0')
    patch.set_edgecolor('#3569a8')
for median in box['medians']:
    median.set_color('#b75252')
    median.set_linewidth(2)
ax.set_title('Temperatur nach Turnier', loc='left', fontweight='bold')
ax.set_xlabel('Turnier')
ax.set_ylabel('Temperatur in Grad C')
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', rotation=20)
fig.tight_layout()
fig.savefig(TEMPERATURE_BY_TOURNAMENT_PATH, bbox_inches='tight')
plt.show()

## Anteil Spiele mit Regen

Der Regenanteil wird je Turnier berechnet. Der horizontale Vergleichswert zeigt den Regenanteil ueber alle Spiele.

In [ ]:
rain_by_tournament = (
    weather_df.groupby('short_name', sort=False)['rain_flag']
    .mean()
    .mul(100)
    .reset_index(name='rain_share_pct')
)
overall_rain_share = weather_df['rain_flag'].mean() * 100

fig, ax = plt.subplots(figsize=(11, 6), dpi=160)
bars = ax.bar(rain_by_tournament['short_name'], rain_by_tournament['rain_share_pct'], color='#2f7d58')
ax.axhline(overall_rain_share, color='#d1812c', linewidth=2, linestyle='--', label=f'Gesamt: {overall_rain_share:.1f}%')
ax.set_title('Anteil Spiele mit Regen je Turnier', loc='left', fontweight='bold')
ax.set_xlabel('Turnier')
ax.set_ylabel('Spiele mit Regen in %')
ax.set_ylim(0, max(25, rain_by_tournament['rain_share_pct'].max() + 8))
ax.grid(axis='y', alpha=0.25)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', rotation=20)
ax.legend(frameon=False)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, height + 0.8, f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

fig.tight_layout()
fig.savefig(RAIN_SHARE_PATH, bbox_inches='tight')
plt.show()

## Output-Checks

Die Checks stellen sicher, dass alle geforderten BD-19-Artefakte geschrieben wurden.

In [ ]:
figure_paths = [
    TEMPERATURE_HIST_PATH,
    FEELS_LIKE_HIST_PATH,
    TEMPERATURE_BY_TOURNAMENT_PATH,
    RAIN_SHARE_PATH,
]

for path in figure_paths:
    assert path.exists(), f'Missing expected figure: {path}'
    assert path.stat().st_size > 1_000, f'Figure looks too small or empty: {path}'

pd.DataFrame({
    'figure': [str(path.relative_to(ROOT)) for path in figure_paths],
    'bytes': [path.stat().st_size for path in figure_paths],
})

## Interpretation fuer BD-19

Die Temperaturwerte reichen von kuehlen bis heissen Bedingungen und unterscheiden sich sichtbar zwischen Turnieren. Regen kommt seltener vor als trockene Spiele, ist aber in mehreren Turnieren vorhanden. Damit ist Wetter als Kontextfaktor sinnvoll analysierbar, besonders Temperatur und gefuehlte Temperatur; Regen sollte wegen des niedrigeren Anteils vorsichtig interpretiert werden.

In [ ]:
interpretation = {
    'n_matches': int(len(weather_df)),
    'temperature_min_c': round(float(weather_df['temperature_c'].min()), 1),
    'temperature_median_c': round(float(weather_df['temperature_c'].median()), 1),
    'temperature_max_c': round(float(weather_df['temperature_c'].max()), 1),
    'feels_like_min_c': round(float(weather_df['feels_like_c'].min()), 1),
    'feels_like_median_c': round(float(weather_df['feels_like_c'].median()), 1),
    'feels_like_max_c': round(float(weather_df['feels_like_c'].max()), 1),
    'rain_match_share_pct': round(float(weather_df['rain_flag'].mean() * 100), 1),
    'rain_matches': int(weather_df['rain_flag'].sum()),
}
interpretation